# Spike: CNN vs CRNN vs LSTM on fold 1, all training clips (preliminary)

**Question.** Does reading the log-mel time frames *in order* help classify vessel types? Three models:
- **CNN**: convolution layers, then an average over time (ignores order).
- **CRNN**: the same convolution layers, then an LSTM over time (uses order).
- **LSTM**: an LSTM alone, reading the raw log-mel frames in order (order only, no convolution).

**This is the full-data copy of `notebooks/spike/spike_cnn_crnn_lstm.ipynb`.** It changes two things and nothing else:
1. **All fold 1 training and validation clips** instead of 600 and 150 per class.
2. **Class-weighted loss**, because the full training set is unbalanced across classes.

**Setup, kept small for the 2026-10-22 review:**
- UniqueShip 12-class, **fold 1 only**, official train / validation / test split (vessel-disjoint).
- All fold 1 training and validation clips; the **whole fold 1 test split** is used for scoring.
- Log-mel 64 × 313 at 32 kHz (same settings as `configs/default.yaml`).
- CNN and CRNN have **identical convolution layers**; they differ only in how the 78 time steps are combined. The LSTM-only model shows what a pure sequence model does without convolution.
- Same optimizer, epochs and batch size for all three; **3 seeds each**, so we can see how much a score moves by chance.

**Preliminary:** one fold, short training, no unit tests. The real five-fold comparison is slice S15 (v1.0, 2026-12-02).

**Runtime on Kaggle (GPU):** about 15 min to join the split zip, about 45 min for features, then about 2 to 3 hours of training (9 runs).

## 1. Settings

In [ ]:
import glob
import io
import json
import os
import posixpath
import random
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd

LOCAL_ZIP = os.environ.get("SPIKE_ZIP")  # only for a local smoke test on a tiny fake bundle
TRAIN_PER_CLASS = None  # all training clips (the capped spike used 600)
VAL_PER_CLASS = None  # all validation clips (the capped spike used 150)
CLASS_WEIGHTED = True  # inverse class frequency; the capped spike was balanced by construction
EPOCHS = int(os.environ.get("SPIKE_EPOCHS", 25))
SEEDS = [0, 1, 2]
BATCH = 64

# Same audio and feature settings as configs/default.yaml
SR, CLIP_S, N_FFT, HOP, N_MELS = 32000, 5.0, 2048, 512, 64
NINE = ["background", "cargo", "fishing", "other", "passengership",
        "pleasurecraft", "sailing", "tanker", "tug"]

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "spike_full_out"
os.makedirs(OUT, exist_ok=True)

## 2. Open the bundle

The dataset is a split zip (`.z01` + `.zip`). `zip -s 0` joins it into one normal zip on Kaggle's temporary disk. We then read **only the clips we need** straight from the zip, instead of unzipping all 26 GB.

In [ ]:
if LOCAL_ZIP:
    ZIP = LOCAL_ZIP
else:
    ZIP = "/tmp/joined.zip"
    if not os.path.exists(ZIP + ".done"):
        hits = glob.glob("/kaggle/input/**/allclass_5hours.z01", recursive=True)
        assert hits, "Attach the uniqueship-12-class-5-hours dataset (Add Input)."
        D = os.path.dirname(hits[0])
        assert os.path.exists(f"{D}/allclass_5hours.zip.part"), "Use a dataset version with both zip parts."
        os.makedirs("/tmp/parts", exist_ok=True)
        for src, dst in [("allclass_5hours.z01", "allclass_5hours.z01"),
                         ("allclass_5hours.zip.part", "allclass_5hours.zip")]:
            if not os.path.lexists(f"/tmp/parts/{dst}"):
                os.symlink(f"{D}/{src}", f"/tmp/parts/{dst}")
        t = time.time()
        subprocess.run("cd /tmp && rm -f joined.zip && zip -q -s 0 parts/allclass_5hours.zip --out joined.zip",
                       shell=True, check=True)
        open(ZIP + ".done", "w").close()
        print(f"joined in {time.time() - t:.0f} s")

zf = zipfile.ZipFile(ZIP)
csv_names = {s: next(n for n in zf.namelist() if n.endswith(f"fold_1/metadata_5s_{s}.csv"))
             for s in ("train", "validation", "test")}
ROOT = csv_names["train"].rsplit("fold_1/", 1)[0]  # e.g. "allclass_5hours/"
print("bundle root inside the zip:", repr(ROOT))

## 3. Choose the clips

- **Group** = the vessel (MMSI); background has no vessel, so it is grouped by recording day. Same rule as the project's leakage test.
- **Round-robin over vessels:** shuffle, number each vessel's clips 0, 1, 2…, then take all clip-0s first, then clip-1s, and so on, up to the cap. This spreads the cap over as many different ships as possible (cargo has ~1 clip per ship, pilotvessel has one ship).
- In this copy nothing is capped: train, validation and test are the whole fold 1 splits.

In [ ]:
def read_split(split):
    df = pd.read_csv(zf.open(csv_names[split]), dtype={"MMSI": str, "date": str}, low_memory=False)
    df["group"] = np.where(df.label == "background", "day:" + df.date, "mmsi:" + df.MMSI)
    df["member"] = [posixpath.normpath(f"{ROOT}fold_1/{p}") for p in df.path]
    return df


def pick(df, per_class, seed=0):
    if per_class is None:
        return df.reset_index(drop=True)
    df = df.sample(frac=1, random_state=seed)
    df["turn"] = df.groupby("group").cumcount()
    return df.sort_values("turn", kind="stable").groupby("label").head(per_class).reset_index(drop=True)


splits = {
    "train": pick(read_split("train"), TRAIN_PER_CLASS),
    "validation": pick(read_split("validation"), VAL_PER_CLASS),
    "test": pick(read_split("test"), None),
}
CLASSES = sorted(splits["train"].label.unique())
for name, df in splits.items():
    assert df.member.is_unique
summary = pd.DataFrame({
    f"{name} clips": df.groupby("label").size() for name, df in splits.items()
} | {
    f"{name} groups": df.groupby("label").group.nunique() for name, df in splits.items()
}).reindex(CLASSES).fillna(0).astype(int)
print(summary.to_string())

## 4. Log-mel features

For each clip: read the WAV from the zip, pad to 5 s if shorter (minimum is 4 s), compute a 64-band mel spectrogram (FFT 2048, hop 512 → 313 frames) and convert power to dB. No per-clip normalization, so loudness differences between vessels are kept. The model input is then standardized with one mean and one standard deviation taken from the training clips.

In [ ]:
import librosa
import soundfile as sf

N = int(SR * CLIP_S)


def logmel(member):
    x, sr = sf.read(io.BytesIO(zf.read(member)), dtype="float32")
    assert sr == SR, (member, sr)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = np.pad(x[:N], (0, max(0, N - len(x))))
    m = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(m, ref=1.0, top_db=None).astype(np.float16)  # (64, 313)


features = {}
for name, df in splits.items():
    t = time.time()
    features[name] = np.stack([logmel(m) for m in df.member])
    print(f"{name}: {features[name].shape} in {time.time() - t:.0f} s")

MU = float(features["train"].astype(np.float32).mean())
SD = float(features["train"].astype(np.float32).std())
print(f"train mean {MU:.1f} dB, std {SD:.1f} dB")

## 5. The three models

Both start with the **same four convolution blocks** (conv 3×3 → batch norm → ReLU → max pool). The first two pools halve frequency and time; the last two halve frequency only, so **78 time steps** survive. Averaging over the 4 remaining mel rows gives a sequence of 78 steps × 128 features (one step ≈ 64 ms of audio).

| | how the 78 steps become one vector | sees the order of events? |
|---|---|---|
| **CNN** | average over time | no |
| **CRNN** | bidirectional LSTM (64 units each way) reads the steps forwards and backwards; its two final states are joined | yes |

**LSTM alone:** a bidirectional LSTM (128 units each way) reads the 313 raw log-mel frames (64 values each) in order, with no convolution. It sees the order of events but has to learn frequency patterns from flat 64-value columns.

In all three, dropout and one linear layer give the 12 class scores. CNN vs CRNN isolates the effect of reading the order; CRNN vs LSTM shows what the convolution front end adds.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def block(c_in, c_out, pool):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(),
        nn.MaxPool2d(pool),
    )


class Net(nn.Module):
    def __init__(self, n_classes, time_pooling):
        super().__init__()
        self.time_pooling = time_pooling
        self.conv = nn.Sequential(
            block(1, 32, (2, 2)), block(32, 64, (2, 2)), block(64, 128, (2, 1)), block(128, 128, (2, 1))
        )
        if time_pooling == "lstm":
            self.lstm = nn.LSTM(128, 64, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):  # x: (batch, 64 mels, 313 frames)
        h = self.conv(x.unsqueeze(1))  # (batch, 128 channels, 4 mel rows, 78 steps)
        h = h.mean(dim=2).transpose(1, 2)  # (batch, 78 steps, 128 features)
        if self.time_pooling == "mean":
            z = h.mean(dim=1)
        else:
            _, (h_n, _) = self.lstm(h)  # h_n: (2 directions, batch, 64)
            z = torch.cat([h_n[0], h_n[1]], dim=1)
        return self.head(self.drop(z))


class LSTMOnly(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.lstm = nn.LSTM(N_MELS, 128, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):  # x: (batch, 64 mels, 313 frames)
        _, (h_n, _) = self.lstm(x.transpose(1, 2))  # reads 313 steps of 64 values
        return self.head(self.drop(torch.cat([h_n[0], h_n[1]], dim=1)))


MODELS = {"CNN": lambda: Net(len(CLASSES), "mean"),
          "CRNN": lambda: Net(len(CLASSES), "lstm"),
          "LSTM": lambda: LSTMOnly(len(CLASSES))}
for name, make in MODELS.items():
    print(f"{name:4s}: {sum(p.numel() for p in make().parameters()):,} parameters")

## 6. Training and scoring

- AdamW, learning rate 1e-3 with cosine decay, batch 64, cross-entropy **weighted by inverse class frequency** (each class counts equally in the loss, however many clips it has).
- After each epoch we score **validation** macro-F1 and keep the best epoch. The **test** split is touched once, at the end.
- **12-class macro-F1** averages over the classes present in fold 1 test; **9-class** averages over the nine classes with enough vessels (rescue, dredger, pilotvessel excluded), from the same predictions. Same rules as the project's evaluation.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

idx = {c: i for i, c in enumerate(CLASSES)}
X = {k: torch.from_numpy(v) for k, v in features.items()}
Y = {k: torch.tensor([idx[c] for c in splits[k].label]) for k in splits}


def scores(y_true, y_pred):
    present = sorted(set(y_true.tolist()))
    nine = [idx[c] for c in NINE if c in idx and idx[c] in present]
    return {
        "f1_12": f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
        "f1_9": f1_score(y_true, y_pred, labels=nine, average="macro", zero_division=0),
        "accuracy": float((y_true == y_pred).float().mean()),
    }


@torch.no_grad()
def logits(model, x):
    model.eval()
    out = [model(((x[i:i + 256].float() - MU) / SD).to(DEV)).cpu() for i in range(0, len(x), 256)]
    return torch.cat(out)


def predict(model, x):
    return logits(model, x).argmax(1)


counts = torch.bincount(Y["train"], minlength=len(CLASSES)).float()
W = (counts.sum() / (len(CLASSES) * counts.clamp(min=1))).to(DEV) if CLASS_WEIGHTED else None
print("class weights:", {c: round(float(w), 2) for c, w in zip(CLASSES, W)} if CLASS_WEIGHTED else "none")


def train(name, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = MODELS[name]().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    gen = torch.Generator().manual_seed(seed)
    best_f1, best_state, history = -1.0, None, []
    t = time.time()
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(X["train"]), generator=gen)
        total = 0.0
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            xb = ((X["train"][b].float() - MU) / SD).to(DEV)
            loss = F.cross_entropy(model(xb), Y["train"][b].to(DEV), weight=W)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        sched.step()
        val = scores(Y["validation"], predict(model, X["validation"]))
        history.append({"epoch": epoch + 1, "loss": total / len(order), "val_f1_12": val["f1_12"]})
        if val["f1_12"] > best_f1:
            best_f1 = val["f1_12"]
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    test_logits = logits(model, X["test"])
    pred = test_logits.argmax(1)
    best_epoch = max(history, key=lambda h: h["val_f1_12"])["epoch"]
    return {
        "model": name, "seed": seed, "best_epoch": best_epoch,
        "val_f1_12": best_f1, **{f"test_{k}": v for k, v in scores(Y["test"], pred).items()},
        "train_seconds": time.time() - t, "history": history, "pred": pred.tolist(),
        "test_logits": test_logits.numpy(),
    }

## 7. Run the three models with three seeds each

In [ ]:
runs = []
for name in MODELS:
    for seed in SEEDS:
        r = train(name, seed)
        runs.append(r)
        print(f"{r['model']:4s} seed {seed}: test 12-class F1 {r['test_f1_12']:.3f}, "
              f"9-class {r['test_f1_9']:.3f}, best epoch {r['best_epoch']}, {r['train_seconds']:.0f} s")

## 8. Results

Mean ± standard deviation over the three seeds. The seed spread shows how much a score moves by chance alone; a gap between the models smaller than that is not a real difference.

In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame([{k: v for k, v in r.items() if k not in ("history", "pred", "test_logits")} for r in runs])
cols = ["test_f1_12", "test_f1_9", "test_accuracy", "val_f1_12", "train_seconds"]
agg = table.groupby("model")[cols].agg(["mean", "std"]).round(3)
print(agg.to_string())
table.to_csv(f"{OUT}/runs.csv", index=False)

# Per-class F1, averaged over seeds
test_classes = sorted(set(Y["test"].tolist()))
per_class = {}
for name in MODELS:
    f = [f1_score(Y["test"], r["pred"], labels=test_classes, average=None, zero_division=0)
         for r in runs if r["model"] == name]
    per_class[name] = np.mean(f, axis=0)
pc = pd.DataFrame(per_class, index=[CLASSES[i] for i in test_classes]).round(3)
pc["test vessels/days"] = splits["test"].groupby("label").group.nunique().reindex(pc.index)
pc["test clips"] = splits["test"].groupby("label").size().reindex(pc.index)
pc["CRNN - CNN"] = (pc.CRNN - pc.CNN).round(3)
pc["LSTM - CNN"] = (pc.LSTM - pc.CNN).round(3)
print("\nPer-class test F1 (mean of 3 seeds):")
print(pc.to_string())
pc.to_csv(f"{OUT}/per_class.csv")

# Confusion matrices (seed 0), rows normalized to recall
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, name in zip(axes, MODELS):
    r = next(r for r in runs if r["model"] == name and r["seed"] == SEEDS[0])
    cm = confusion_matrix(Y["test"], r["pred"], labels=range(len(CLASSES))).astype(float)
    cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            if cm[i, j] >= 0.05:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if cm[i, j] > 0.5 else "black")
    ax.set_title(f"{name}: fold 1 test (seed {SEEDS[0]}), rows = true class")
    ax.set_xlabel("predicted")
plt.tight_layout()
plt.savefig(f"{OUT}/confusion.png", dpi=150)
plt.show()

# Validation curves
fig, ax = plt.subplots(figsize=(8, 4.5))
for r in runs:
    color = {"CNN": "tab:blue", "CRNN": "tab:orange", "LSTM": "tab:green"}[r["model"]]
    ax.plot([h["val_f1_12"] for h in r["history"]], color=color,
            alpha=0.8, label=f"{r['model']} seed {r['seed']}")
ax.set_xlabel("epoch")
ax.set_ylabel("validation 12-class macro-F1")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/val_curves.png", dpi=150)
plt.show()

json.dump({
    "settings": {"train_per_class": TRAIN_PER_CLASS, "val_per_class": VAL_PER_CLASS,
                 "class_weighted": CLASS_WEIGHTED, "epochs": EPOCHS,
                 "seeds": SEEDS, "classes": CLASSES},
    "clips": {k: len(v) for k, v in splits.items()},
    "summary": {m: {c: [float(agg.loc[m, (c, "mean")]), float(agg.loc[m, (c, "std")])] for c in cols}
                for m in agg.index},
}, open(f"{OUT}/results.json", "w"), indent=2)
print("saved to", OUT)

## 9. Rescore on the paper's 5 classes

The dataset paper reports 66.5% accuracy on 5 classes: background, cargo, passengership, tanker, tug. Our models are trained on 12 classes, so this is a **rescore, not a 5-class model**:

- keep only the test clips whose true class is one of the 5;
- from each clip's 12 scores, keep the 5 and predict the highest of those.

Our fold 1 test split is not balanced across the 5 classes, unlike the paper's balanced bundle, so **balanced accuracy** (the mean of the 5 per-class recalls) is reported beside plain accuracy. Still not directly comparable to the paper: 12-class training, less data, no pretraining.

The test logits of every run are saved to `test_logits.npz`, so other subsets can be rescored later without retraining.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

FIVE = ["background", "cargo", "passengership", "tanker", "tug"]
five_idx = [idx[c] for c in FIVE]
y_test = Y["test"].numpy()
keep = np.isin(y_test, five_idx)
y5 = y_test[keep]

five_rows = []
for r in runs:
    sub = r["test_logits"][keep][:, five_idx]
    p5 = np.array(five_idx)[sub.argmax(1)]
    five_rows.append({
        "model": r["model"], "seed": r["seed"],
        "accuracy_5": float((p5 == y5).mean()),
        "balanced_accuracy_5": float(balanced_accuracy_score(y5, p5)),
        "f1_5": float(f1_score(y5, p5, labels=five_idx, average="macro", zero_division=0)),
    })
five = pd.DataFrame(five_rows)
five_cols = ["accuracy_5", "balanced_accuracy_5", "f1_5"]
five_agg = five.groupby("model")[five_cols].agg(["mean", "std"]).round(3)
print(f"{keep.sum()} test clips in the 5 classes:")
print(splits["test"].label[keep].value_counts().reindex(FIVE).to_string())
print()
print(five_agg.to_string())
five.to_csv(f"{OUT}/five_class.csv", index=False)

np.savez_compressed(
    f"{OUT}/test_logits.npz",
    logits=np.stack([r["test_logits"] for r in runs]),
    models=np.array([r["model"] for r in runs]),
    seeds=np.array([r["seed"] for r in runs]),
    y_true=y_test,
    classes=np.array(CLASSES),
    members=splits["test"].member.to_numpy(),
)

results = json.load(open(f"{OUT}/results.json"))
results["five_class"] = {
    "classes": FIVE,
    "test_clips": int(keep.sum()),
    "summary": {m: {c: [float(five_agg.loc[m, (c, "mean")]), float(five_agg.loc[m, (c, "std")])]
                    for c in five_cols} for m in five_agg.index},
}
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
print("saved five_class.csv, test_logits.npz and results.json[five_class] to", OUT)